In [1]:
from manim import *

In [5]:
%%manim -qh -v WARNING AnalysensHuvudsats
from manim import *
import numpy as np

class AnalysensHuvudsats(Scene):
    def construct(self):
        # 0. Rubrik synlig direkt i första bildrutan
        title = Text("Analysens huvudsats", font_size=26, color=WHITE).to_corner(UR, buff=0.35)
        title_line = Line(title.get_left(), title.get_right(), color=GRAY_B, stroke_width=1.5).next_to(
            title, DOWN, buff=0.08
        )
        self.add(title, title_line)

        # 1. Godtycklig, mjukt vågig funktion f(t) och dess exakta areafunktion A(x) från a
        a = 0.8
        x_end = 3.8
        x_inspect = 2.7  # Punkt där vi pausar och zoomar in på dx och tangenten (kallas x_1)

        def f(t):
            return 1.1 + 0.45 * np.sin(1.4 * t) + 0.12 * t

        def A_raw(x):
            return 1.1 * x - (0.45 / 1.4) * np.cos(1.4 * x) + 0.06 * x**2

        offset = A_raw(a)
        def A(x):
            return A_raw(x) - offset

        # 2. Koordinatsystem i exakt samma layout som Scen 1
        ax_cfg = {
            "include_tip": True,
            "stroke_width": 2,
            "tip_width": 0.12,
            "tip_height": 0.12,
            "include_ticks": False
        }

        ax_top = Axes(
            x_range=[0, 4.5, 1], y_range=[0, 4.5, 1],
            x_length=5.0, y_length=2.85,
            axis_config=ax_cfg
        ).move_to(LEFT * 2.6 + UP * 1.8)

        ax_bot = Axes(
            x_range=[0, 4.5, 1], y_range=[0, 2.5, 1],
            x_length=5.0, y_length=2.55,
            axis_config=ax_cfg
        ).move_to(LEFT * 2.6 + DOWN * 1.85)

        # Axelrubriker (t på horisontella axlarna, y = A(x) och y = f(t) på vertikala)
        ax_top_y_lbl = MathTex(r"y = A(x) = \int_a^x f(t)\,dt", font_size=24, color=BLUE).next_to(
            ax_top.y_axis.get_end(), UP, buff=0.1
        ).shift(RIGHT * 0.95)
        ax_top_x_lbl = MathTex("t", font_size=24).next_to(ax_top.x_axis.get_end(), RIGHT, buff=0.08)

        ax_bot_y_lbl = MathTex("y = f(t)", font_size=26, color=YELLOW).next_to(
            ax_bot.y_axis.get_end(), UP, buff=0.1
        ).shift(RIGHT * 0.25)
        ax_bot_x_lbl = MathTex("t", font_size=24).next_to(ax_bot.x_axis.get_end(), RIGHT, buff=0.08)

        graph_f = ax_bot.plot(f, x_range=[0.3, 4.2], color=YELLOW, stroke_width=3)

        # Fasta markeringar för startpunkten a
        a_marks = VGroup()
        for ax, func in [(ax_top, A), (ax_bot, f)]:
            tick = ax.x_axis.get_tick(a, size=0.08)
            lbl = MathTex("a", font_size=24).next_to(ax.c2p(a, 0), DOWN, buff=0.1)
            a_marks.add(tick, lbl)
        a_marks.add(DashedLine(ax_bot.c2p(a, 0), ax_bot.c2p(a, f(a)), color=GRAY, stroke_width=1.5))

        self.play(
            Create(ax_top), Create(ax_bot),
            Write(ax_top_y_lbl), Write(ax_top_x_lbl),
            Write(ax_bot_y_lbl), Write(ax_bot_x_lbl)
        )
        self.play(Create(graph_f), FadeIn(a_marks))
        self.wait(0.5)

        # 3. Dynamisk animation där x glider från a till x_inspect
        x_tracker = ValueTracker(a + 0.05)

        area_shaded = always_redraw(lambda: ax_bot.get_area(
            graph_f, x_range=[a, x_tracker.get_value()], color=BLUE, opacity=0.45
        ))

        def area_label_pos(x_val):
            x_mid = 0.5 * (a + x_val)
            y_mid = 0.45 * min(f(a), f(x_val), f(x_mid))
            return ax_bot.c2p(x_mid, max(y_mid, 0.25))

        area_lbl_bot = always_redraw(lambda: MathTex(
            "A(x)", font_size=22, color=WHITE
        ).move_to(area_label_pos(x_tracker.get_value())))

        graph_A_dyn = always_redraw(lambda: ax_top.plot(
            A, x_range=[a, x_tracker.get_value()], color=BLUE, stroke_width=3.2
        ))

        # Rörliga markörer för x på båda x-axlarna + vertikal kopplingslinje
        dot_top = always_redraw(lambda: Dot(
            ax_top.c2p(x_tracker.get_value(), A(x_tracker.get_value())), color=WHITE, radius=0.055
        ))
        dot_bot = always_redraw(lambda: Dot(
            ax_bot.c2p(x_tracker.get_value(), f(x_tracker.get_value())), color=YELLOW, radius=0.055
        ))

        v_link = always_redraw(lambda: DashedLine(
            ax_bot.c2p(x_tracker.get_value(), 0),
            ax_top.c2p(x_tracker.get_value(), A(x_tracker.get_value())),
            color=GRAY_B, stroke_width=1.4
        ))

        x_lbl_top = always_redraw(lambda: MathTex("x", font_size=24, color=WHITE).next_to(
            ax_top.c2p(x_tracker.get_value(), 0), DOWN, buff=0.1
        ))
        x_lbl_bot = always_redraw(lambda: MathTex("x", font_size=24, color=WHITE).next_to(
            ax_bot.c2p(x_tracker.get_value(), 0), DOWN, buff=0.1
        ))

        # Rörlig höjdmarkering A(x) på övre y-axeln
        y_dash_top = always_redraw(lambda: DashedLine(
            ax_top.c2p(0, A(x_tracker.get_value())),
            ax_top.c2p(x_tracker.get_value(), A(x_tracker.get_value())),
            color=BLUE_B, stroke_width=1.4
        ))
        y_lbl_top = always_redraw(lambda: MathTex("A(x)", font_size=22, color=BLUE_B).next_to(
            ax_top.c2p(0, A(x_tracker.get_value())), LEFT, buff=0.1
        ))

        self.add(
            area_shaded, area_lbl_bot, graph_A_dyn,
            v_link, y_dash_top, dot_top, dot_bot, x_lbl_top, x_lbl_bot, y_lbl_top
        )

        # Kör fram till x_inspect (x_1)
        self.play(x_tracker.animate.set_value(x_inspect), run_time=3.2, rate_func=smooth)
        self.wait(0.5)

        # 4. Pausa vid x_1 och visa en liten ökning dx -> dA ≈ f(x_1)·dx (ligger kvar genom resten)
        dx = 0.35
        x1 = x_inspect
        x2 = x1 + dx
        y1 = A(x1)
        y2 = A(x2)
        fx1 = f(x1)

        # Fast markering för x_1 på båda x-axlarna
        tick_x1_top = ax_top.x_axis.get_tick(x1, size=0.08)
        tick_x1_bot = ax_bot.x_axis.get_tick(x1, size=0.08)
        lbl_x1_top = MathTex("x_1", font_size=22, color=WHITE).next_to(ax_top.c2p(x1, 0), DOWN, buff=0.1)
        lbl_x1_bot = MathTex("x_1", font_size=22, color=WHITE).next_to(ax_bot.c2p(x1, 0), DOWN, buff=0.1)

        # Liten grön remsa under f(t) med "dx" inuti
        strip = Rectangle(
            width=ax_bot.x_axis.unit_size * dx,
            height=ax_bot.y_axis.unit_size * fx1,
            fill_color=GREEN, fill_opacity=0.6,
            stroke_color=GREEN_B, stroke_width=2
        ).move_to(ax_bot.c2p(x1 + dx / 2, fx1 / 2))
        lbl_dx_inside = MathTex("dA", font_size=18, color=WHITE).move_to(strip.get_center())

        dash_fx1 = DashedLine(ax_bot.c2p(0, fx1), ax_bot.c2p(x1, fx1), color=YELLOW, stroke_width=1.5)
        tick_fx1 = ax_bot.y_axis.get_tick(fx1, size=0.08).set_color(YELLOW)
        lbl_fx1_y = MathTex("f(x_1)", font_size=22, color=YELLOW).next_to(ax_bot.c2p(0, fx1), LEFT, buff=0.1)

        # Motsvarande steg dx och dA på A(x)
        step_dx_top = Line(ax_top.c2p(x1, y1), ax_top.c2p(x2, y1), color=WHITE, stroke_width=2)
        step_dA_top = Line(ax_top.c2p(x2, y1), ax_top.c2p(x2, y2), color=GREEN, stroke_width=4)
        lbl_dx_top = MathTex("dx", font_size=20).next_to(step_dx_top, DOWN, buff=0.06)
        lbl_dA_top = MathTex("dA", font_size=22, color=GREEN).next_to(step_dA_top, RIGHT, buff=0.08)

        # Tangent vid punkten (x_1, A(x_1)) med lutning f(x_1)
        tangent = ax_top.plot(
            lambda t: y1 + fx1 * (t - x1),
            x_range=[x1 - 0.55, x1 + 0.65],
            color=RED_B, stroke_width=3
        )
        lbl_tan = MathTex(r"\text{Lutning } A'(x_1) = f(x_1)", font_size=21, color=RED_B).next_to(
            ax_top.c2p(x1 + 0.35, y1 + fx1 * 0.35), UL, buff=0.15
        )

        self.play(
            FadeIn(tick_x1_top, tick_x1_bot, lbl_x1_top, lbl_x1_bot),
            Create(dash_fx1), FadeIn(tick_fx1, lbl_fx1_y),
            FadeIn(strip), Write(lbl_dx_inside)
        )
        self.wait(0.6)

        self.play(
            ReplacementTransform(strip.copy(), step_dA_top),
            Create(step_dx_top),
            Write(lbl_dx_top), Write(lbl_dA_top)
        )
        self.wait(0.6)

        self.play(Create(tangent), Write(lbl_tan))
        self.wait(1.0)

        # Fortsätt rörelsen till x_end medan inspektionsförklaringarna ligger kvar
        self.play(x_tracker.animate.set_value(x_end), run_time=2.0, rate_func=smooth)
        self.wait(0.5)

        # 5. Matematisk sammanfattning i högerspalten
        summary = VGroup(
            MathTex(r"\text{1. Areaökning: } dA \approx f(x_1)\,dx", font_size=24, color=GREEN_A),
            MathTex(r"\text{2. Ändringskvot: } \frac{dA}{dx} \approx f(x_1)", font_size=24, color=WHITE),
            MathTex(r"\text{3. Då } dx \to 0 \text{ (tangentens lutning):}", font_size=23, color=RED_B),
            MathTex(r"A'(x_1) = f(x_1)", font_size=28, color=YELLOW),
            Line(LEFT * 2.1, RIGHT * 2.1, color=GRAY),
            MathTex(r"\frac{d}{dx}\int_a^x f(t)\,dt = f(x)", font_size=28, color=BLUE_B)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.24).move_to(RIGHT * 3.6 + DOWN * 0.25)

        for line in summary:
            self.play(Write(line), run_time=0.6)
            self.wait(0.25)

        self.wait(2.5)

Manim Community v0.22.1